# Disaster Recovery Cost Prediction and Resilience Optimization
## Week 1 Day 2 — Understanding the FEMA Data Source

Section A — Imports and setup

In [3]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

BASE_URL = "https://www.fema.gov/api/open"

ENDPOINTS = {
    "declarations": f"{BASE_URL}/v2/DisasterDeclarationsSummaries",
    "public_assistance": f"{BASE_URL}/v2/PublicAssistanceFundedProjectsDetails",
    "disaster_summaries": f"{BASE_URL}/v1/FemaWebDisasterSummaries",
}

SAMPLE_DIR = Path("../data/raw/samples")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
#Helper to fetch sample data
def fetch_fema_sample(url: str, top: int = 200, select: str | None = None) -> pd.DataFrame:
    params = {
        "$top": top,
        "$format": "json"
    }
    if select:
        params["$select"] = select

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()

    # OpenFEMA commonly returns records under "DisasterDeclarationsSummaries",
    # "PublicAssistanceFundedProjectsDetails", etc. To stay robust, grab the first list value.
    records = None
    for value in payload.values():
        if isinstance(value, list):
            records = value
            break

    if records is None:
        raise ValueError("No record list found in API response.")

    return pd.DataFrame(records)

In [5]:
# Pull 200 records from each dataset
df_declarations = fetch_fema_sample(ENDPOINTS["declarations"], top=200)
df_pa = fetch_fema_sample(ENDPOINTS["public_assistance"], top=200)
df_summaries = fetch_fema_sample(ENDPOINTS["disaster_summaries"], top=200)

print("Declarations shape:", df_declarations.shape)
print("Public Assistance shape:", df_pa.shape)
print("Disaster Summaries shape:", df_summaries.shape)

Declarations shape: (200, 28)
Public Assistance shape: (200, 25)
Disaster Summaries shape: (200, 14)


In [6]:
# Save raw samples

df_declarations.to_csv(SAMPLE_DIR / "declarations_sample_200.csv", index=False)
df_pa.to_csv(SAMPLE_DIR / "public_assistance_sample_200.csv", index=False)
df_summaries.to_csv(SAMPLE_DIR / "disaster_summaries_sample_200.csv", index=False)

In [7]:
display(df_declarations.head())
display(df_pa.head())
display(df_summaries.head())

,femaDeclarationString,disasterNumber,state,declarationType,declarationDate,fyDeclared,incidentType,declarationTitle,ihProgramDeclared,iaProgramDeclared,paProgramDeclared,hmProgramDeclared,incidentBeginDate,incidentEndDate,disasterCloseoutDate,tribalRequest,fipsStateCode,fipsCountyCode,placeCode,designatedArea,declarationRequestNumber,lastIAFilingDate,incidentId,region,designatedIncidentTypes,lastRefresh,hash,id
0,FM-5529-OR,5529,OR,FM,2024-08-09T00:00:00.000Z,2024,Fire,LEE FALLS FIRE,False,False,True,True,2024-08-08T00:00:00.000Z,None,None,False,41,067,99067,Washington (County),24122,None,2024081001,10,R,2024-08-27T18:22:14.800Z,ae87cf3c6ed795015b714af7166c7c295b2b67c7,09e3f81a-5e16-4b72-b317-1c64e0cfa59c
1,FM-5528-OR,5528,OR,FM,2024-08-06T00:00:00.000Z,2024,Fire,ELK LANE FIRE,False,False,True,True,2024-08-04T00:00:00.000Z,None,None,False,41,031,99031,Jefferson (County),24116,None,2024080701,10,R,2024-08-27T18:22:14.800Z,432cf0995c47e3895cea696ede5621b810460501,59983f89-30bf-4888-b21b-62e8d57d9aac
2,FM-5527-OR,5527,OR,FM,2024-08-02T00:00:00.000Z,2024,Fire,MILE MARKER 132 FIRE,False,False,True,True,2024-08-02T00:00:00.000Z,None,None,False,41,017,99017,Deschutes (County),24111,None,2024080301,10,R,2024-08-27T18:22:14.800Z,2f21d90cb6bc64b0d4121aa3f18d852bbb4b11fa,8d13ecf0-bc2f-496b-8c9f-b2e73da832a0
3,DR-4312-CA,4312,CA,DR,2017-05-02T00:00:00.000Z,2017,Severe Storm,FLOODING,False,False,True,True,2017-02-08T00:00:00.000Z,2017-02-11T00:00:00.000Z,2025-03-25T00:00:00.000Z,True,06,000,60347,Resighini Rancheria (Indian Reservation),17035,None,2017041001,9,None,2025-03-26T20:21:32.579Z,432a3a64bdbb291ae26cf5a27a33deeabb380481,98a7c5bb-2346-45aa-a1ca-0399440d4f0b
4,DR-4251-AL,4251,AL,DR,2016-01-21T00:00:00.000Z,2016,Severe Storm,"SEVERE STORMS, TORNADOES, STRAIGHT-LINE WINDS,...",False,False,True,True,2015-12-23T00:00:00.000Z,2015-12-31T00:00:00.000Z,2025-03-27T00:00:00.000Z,False,01,001,99001,Autauga (County),16003,None,2015122301,4,None,2025-03-27T12:21:46.559Z,dcd4ce6b37ee49875b3f1e32e9a8a16cd6a803d3,5229bbae-eee6-42b8-b277-edbafa8d6cb2


,disasterNumber,declarationDate,incidentType,pwNumber,applicationTitle,applicantId,damageCategoryCode,damageCategoryDescrip,projectStatus,projectProcessStep,projectSize,county,countyCode,stateAbbreviation,stateNumberCode,projectAmount,federalShareObligated,totalObligated,lastObligationDate,firstObligationDate,mitigationAmount,gmProjectId,gmApplicantId,lastRefresh,hash
0,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),1,(PW# 1) IMMEDIATE NEEDS FUNDING,465-19792-00,B,Emergency Protective Measures,Active,Project Closed Out,Large,Val Verde County,465,TX,48,100000.00,75000.00,80340.00,1998-09-15T14:25:07.000Z,1998-09-15T14:25:07.000Z,0,1021769,268458,2025-11-27T15:05:59.253Z,addcfded82ae348f46ff034a4564f983a9dea897
1,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),5,(PW# 5) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,Large,Val Verde County,465,TX,48,19685.50,14764.13,15461.00,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0,1062596,268458,2025-11-27T15:05:59.253Z,05c6c522b930a9c38b52a0c4b0de853e98b4cb75
2,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),7,(PW# 7) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,Large,Val Verde County,465,TX,48,26111.00,19583.25,20507.58,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0,1062598,268458,2025-11-27T15:05:59.253Z,0addbfed02721821348612482bfe36d8fe587d0f
3,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),8,(PW# 8) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,Large,Val Verde County,465,TX,48,26250.83,19688.12,20617.40,1998-09-28T15:02:18.000Z,1998-09-28T15:02:18.000Z,0,1062599,268458,2025-11-27T15:05:59.253Z,d01abd86d6129cc818c5cee0f9373e3097c245e6
4,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),10,(PW# 10) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,Large,Val Verde County,465,TX,48,34502.50,25876.88,27098.27,1998-09-28T15:02:18.000Z,1998-09-28T15:02:18.000Z,0,1062578,268458,2025-11-27T15:05:59.253Z,ae1b100b0fe6a9fd0a75c120e0ca3a5e4be6d356


,disasterNumber,totalNumberIaApproved,totalAmountIhpApproved,totalAmountHaApproved,totalAmountOnaApproved,totalObligatedAmountPa,totalObligatedAmountCatAb,totalObligatedAmountCatC2g,paLoadDate,iaLoadDate,totalObligatedAmountHmgp,hash,lastRefresh,id
0,3601,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,3de68baba960e69da445cf822d3dd859081fb34a,2023-10-09T23:02:26.341Z,faafecca-0f76-4fb8-8ffd-b6f46f3b712c
1,3602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,58566c446fce5cabbd0c3412a6bb3daa4ada1993,2023-10-09T23:02:26.341Z,b74f0dc2-fab5-42b9-acf7-c94df14d85ad
2,1981,7469.0,95754618.67,93631947.01,2122671.66,2.243558e+08,94967791.86,1.197739e+08,2026-02-06T00:00:00.000Z,2026-04-05T00:00:00.000Z,71549737.00,24cc4ca4c5602ae7f9ed7a94ff2ad08b28789674,2026-04-05T06:02:19.627Z,ecb4804b-cfde-4437-93b5-1334c1815fee
3,1802,NaN,NaN,NaN,NaN,1.881275e+07,13423088.38,4.869890e+06,2026-02-06T00:00:00.000Z,None,2710679.00,d5b8865b047c2db34bae425a847682fc5c952d47,2026-02-06T02:02:06.586Z,b3ae5013-d6cc-46f3-a692-18191be44083
4,4317,1978.0,12527583.31,10311780.12,2215803.19,7.731984e+07,11526563.99,6.437977e+07,2026-04-05T00:00:00.000Z,2026-04-05T00:00:00.000Z,17801245.73,a37f2cacbd2f2184de5465c2148b1a228e4be246,2026-04-05T06:02:19.627Z,c3af2a42-752e-427c-a9ae-6b6cefa31b22


In [8]:
# Profiling function

def profile_dataframe(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    profile_rows = []

    for col in df.columns:
        series = df[col]
        non_null = series.dropna()

        row = {
            "dataset": dataset_name,
            "column": col,
            "dtype": str(series.dtype),
            "rows": len(df),
            "null_count": int(series.isna().sum()),
            "null_rate_pct": round(series.isna().mean() * 100, 2),
            "unique_count": int(series.nunique(dropna=True)),
            "sample_values": ", ".join(map(str, non_null.astype(str).head(5).tolist()))
        }

        # numeric range
        if pd.api.types.is_numeric_dtype(series):
            row["min_value"] = non_null.min() if not non_null.empty else np.nan
            row["max_value"] = non_null.max() if not non_null.empty else np.nan
        else:
            row["min_value"] = np.nan
            row["max_value"] = np.nan

        profile_rows.append(row)

    return pd.DataFrame(profile_rows)

In [9]:
# Create profiling tables
profile_declarations = profile_dataframe(df_declarations, "DisasterDeclarationsSummaries")
profile_pa = profile_dataframe(df_pa, "PublicAssistanceFundedProjectsDetails")
profile_summaries = profile_dataframe(df_summaries, "FemaWebDisasterSummaries")

display(profile_declarations)
display(profile_pa)
display(profile_summaries)

,dataset,column,dtype,rows,null_count,null_rate_pct,unique_count,sample_values,min_value,max_value
0,DisasterDeclarationsSummaries,femaDeclarationString,object,200,0,0.0,133,"FM-5529-OR, FM-5528-OR, FM-5527-OR, DR-4312-CA...",NaN,NaN
1,DisasterDeclarationsSummaries,disasterNumber,int64,200,0,0.0,133,"5529, 5528, 5527, 4312, 4251",4251,5529
2,DisasterDeclarationsSummaries,state,object,200,0,0.0,22,"OR, OR, OR, CA, AL",NaN,NaN
3,DisasterDeclarationsSummaries,declarationType,object,200,0,0.0,2,"FM, FM, FM, DR, DR",NaN,NaN
4,DisasterDeclarationsSummaries,declarationDate,object,200,0,0.0,102,"2024-08-09T00:00:00.000Z, 2024-08-06T00:00:00....",NaN,NaN
5,DisasterDeclarationsSummaries,fyDeclared,int64,200,0,0.0,7,"2024, 2024, 2024, 2017, 2016",2016,2024
6,DisasterDeclarationsSummaries,incidentType,object,200,0,0.0,2,"Fire, Fire, Fire, Severe Storm, Severe Storm",NaN,NaN
7,DisasterDeclarationsSummaries,declarationTitle,object,200,0,0.0,132,"LEE FALLS FIRE, ELK LANE FIRE, MILE MARKER 132...",NaN,NaN
8,DisasterDeclarationsSummaries,ihProgramDeclared,bool,200,0,0.0,1,"False, False, False, False, False",False,False
9,DisasterDeclarationsSummaries,iaProgramDeclared,bool,200,0,0.0,1,"False, False, False, False, False",False,False


,dataset,column,dtype,rows,null_count,null_rate_pct,unique_count,sample_values,min_value,max_value
0,PublicAssistanceFundedProjectsDetails,disasterNumber,int64,200,0,0.0,24,"1239, 1239, 1239, 1239, 1239",1239.00,3543.00
1,PublicAssistanceFundedProjectsDetails,declarationDate,object,200,0,0.0,14,"1998-08-26T00:00:00.000Z, 1998-08-26T00:00:00....",NaN,NaN
2,PublicAssistanceFundedProjectsDetails,incidentType,object,200,0,0.0,5,"Severe Storm(s), Severe Storm(s), Severe Storm...",NaN,NaN
3,PublicAssistanceFundedProjectsDetails,pwNumber,int64,200,0,0.0,100,"1, 5, 7, 8, 10",1.00,15802.00
4,PublicAssistanceFundedProjectsDetails,applicationTitle,object,200,0,0.0,185,"(PW# 1) IMMEDIATE NEEDS FUNDING, (PW# 5) Not P...",NaN,NaN
5,PublicAssistanceFundedProjectsDetails,applicantId,object,200,0,0.0,111,"465-19792-00, 465-19792-00, 465-19792-00, 465-...",NaN,NaN
6,PublicAssistanceFundedProjectsDetails,damageCategoryCode,object,200,0,0.0,7,"B, G, G, G, G",NaN,NaN
7,PublicAssistanceFundedProjectsDetails,damageCategoryDescrip,object,200,0,0.0,8,"Emergency Protective Measures, Parks, Recreati...",NaN,NaN
8,PublicAssistanceFundedProjectsDetails,projectStatus,object,200,0,0.0,2,"Active, Active, Active, Active, Active",NaN,NaN
9,PublicAssistanceFundedProjectsDetails,projectProcessStep,object,200,0,0.0,3,"Project Closed Out, Project Closed Out, Projec...",NaN,NaN


,dataset,column,dtype,rows,null_count,null_rate_pct,unique_count,sample_values,min_value,max_value
0,FemaWebDisasterSummaries,disasterNumber,int64,200,0,0.0,200,"3601, 3602, 1981, 1802, 4317",1255.00,5.614000e+03
1,FemaWebDisasterSummaries,totalNumberIaApproved,float64,200,186,93.0,14,"7469.0, 1978.0, 2059.0, 8340.0, 25401.0",164.00,2.540100e+04
2,FemaWebDisasterSummaries,totalAmountIhpApproved,float64,200,186,93.0,14,"95754618.67, 12527583.31, 9506708.04, 57084521...",549480.66,1.035041e+08
3,FemaWebDisasterSummaries,totalAmountHaApproved,float64,200,188,94.0,12,"93631947.01, 10311780.12, 8474203.73, 93779140...",447761.87,9.377914e+07
4,FemaWebDisasterSummaries,totalAmountOnaApproved,float64,200,186,93.0,14,"2122671.66, 2215803.19, 1032504.31, 57084521.7...",101718.79,5.708452e+07
5,FemaWebDisasterSummaries,totalObligatedAmountPa,float64,200,37,18.5,163,"224355761.7, 18812748.86, 77319838.79, 2981057...",154.00,4.732560e+09
6,FemaWebDisasterSummaries,totalObligatedAmountCatAb,float64,200,49,24.5,151,"94967791.86, 13423088.38, 11526563.99, 1436026...",154.00,2.178073e+09
7,FemaWebDisasterSummaries,totalObligatedAmountCatC2g,float64,200,118,59.0,82,"119773860.39, 4869890.36, 64379774.1, 15048520...",0.00,2.408879e+08
8,FemaWebDisasterSummaries,paLoadDate,object,200,37,18.5,2,"2026-02-06T00:00:00.000Z, 2026-02-06T00:00:00....",NaN,NaN
9,FemaWebDisasterSummaries,iaLoadDate,object,200,186,93.0,1,"2026-04-05T00:00:00.000Z, 2026-04-05T00:00:00....",NaN,NaN


In [10]:
# Optional date conversion check
date_candidate_cols = [
    "declarationDate", "incidentBeginDate", "incidentEndDate"
]

for col in date_candidate_cols:
    if col in df_declarations.columns:
        df_declarations[col] = pd.to_datetime(df_declarations[col], errors="coerce")

df_declarations[date_candidate_cols].dtypes if all(c in df_declarations.columns for c in date_candidate_cols) else "Some date columns not found"

declarationDate      datetime64[ns, UTC]
incidentBeginDate    datetime64[ns, UTC]
incidentEndDate      datetime64[ns, UTC]
dtype: object

In [11]:
# Key field inspection
key_candidates = {
    "declarations": ["disasterNumber", "state", "incidentType", "declarationType"],
    "public_assistance": ["project_id", "disasterNumber", "state", "incidentType", "obligatedAmount", "projectCategory"],
    "disaster_summaries": ["disasterNumber", "totalFederalShareObligations", "programBreakdown"]
}

for dataset_name, cols in key_candidates.items():
    print(f"\n--- {dataset_name.upper()} ---")
    df = {
        "declarations": df_declarations,
        "public_assistance": df_pa,
        "disaster_summaries": df_summaries
    }[dataset_name]

    available_cols = [c for c in cols if c in df.columns]
    display(df[available_cols].head(10))


--- DECLARATIONS ---


,disasterNumber,state,incidentType,declarationType
0,5529,OR,Fire,FM
1,5528,OR,Fire,FM
2,5527,OR,Fire,FM
3,4312,CA,Severe Storm,DR
4,4251,AL,Severe Storm,DR
5,4251,AL,Severe Storm,DR
6,4251,AL,Severe Storm,DR
7,4251,AL,Severe Storm,DR
8,4251,AL,Severe Storm,DR
9,5516,OR,Fire,FM



--- PUBLIC_ASSISTANCE ---


,disasterNumber,incidentType
0,1239,Severe Storm(s)
1,1239,Severe Storm(s)
2,1239,Severe Storm(s)
3,1239,Severe Storm(s)
4,1239,Severe Storm(s)
5,1239,Severe Storm(s)
6,1239,Severe Storm(s)
7,1239,Severe Storm(s)
8,1239,Severe Storm(s)
9,1239,Severe Storm(s)



--- DISASTER_SUMMARIES ---


,disasterNumber
0,3601
1,3602
2,1981
3,1802
4,4317
5,1292
6,1255
7,3163
8,1817
9,3187


In [12]:
# Join key checks

def key_presence_check(df_left, df_right, key):
    left_keys = set(df_left[key].dropna().astype(str).unique()) if key in df_left.columns else set()
    right_keys = set(df_right[key].dropna().astype(str).unique()) if key in df_right.columns else set()

    overlap = left_keys.intersection(right_keys)

    return {
        "key": key,
        "left_unique": len(left_keys),
        "right_unique": len(right_keys),
        "overlap_unique": len(overlap)
    }

checks = []
checks.append(key_presence_check(df_declarations, df_pa, "disasterNumber"))
checks.append(key_presence_check(df_declarations, df_summaries, "disasterNumber"))
checks.append(key_presence_check(df_pa, df_summaries, "disasterNumber"))

pd.DataFrame(checks)

,key,left_unique,right_unique,overlap_unique
0,disasterNumber,133,24,0
1,disasterNumber,133,200,13
2,disasterNumber,24,200,1


In [13]:
# Save profiling outputs
REPORT_DIR = Path("../reports/weekly_updates")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

profile_declarations.to_csv(REPORT_DIR / "profile_declarations_day2.csv", index=False)
profile_pa.to_csv(REPORT_DIR / "profile_public_assistance_day2.csv", index=False)
profile_summaries.to_csv(REPORT_DIR / "profile_disaster_summaries_day2.csv", index=False)

### Datasets reviewed
- DisasterDeclarationsSummaries
- PublicAssistanceFundedProjectsDetails
- FemaWebDisasterSummaries

### Likely join keys
- Primary join key: `disasterNumber`
- Secondary validation fields: `state`, `incidentType`

### Early observations
- Public Assistance is at a more granular, project-level detail.
- Disaster Declarations provides event/declaration context.
- FEMA Web Disaster Summaries appears useful for obligation cross-checks.

### Implication for Day 3
Tomorrow’s ingestion pipeline should:
1. fetch these three datasets,
2. preserve raw copies,
3. standardize key columns,
4. aggregate project-level obligations to disaster level,
5. validate joins using `disasterNumber`.